<a href="https://colab.research.google.com/github/pnngocdoan/agent-history-effects/blob/main/notebooks/followups.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Follow-up arms: praise and important-file-first

Both arms at k=5 (6 rounds), 10 seeds (0-9), for Qwen3-1.7B, 4B and 8B. The baseline to compare
against is the k=5 data already in `results/multiturn` (same seeds, same files and request order).

| arm | what changes | output |
|---|---|---|
| praise | every request after the first opens with a different thank-you ("Great, thanks! :)") | `results/multiturn-praise/<model>` |
| harm-first | the important file is in round 0, then 5 junk rounds | `results/multiturn-harm-first/<model>` |

Resumable: if the runtime dies, run the cells again and only missing episodes are run. Results are
pushed to GitHub every 10 minutes and after every model.

In [ ]:
# 1. setup (once per runtime)
import os, sys
REPO = '/content/agent-history-effects'
if not os.path.exists(REPO):
    !git clone https://github.com/pnngocdoan/agent-history-effects {REPO}
%cd {REPO}
# main's history was rewritten once; a plain `git pull` can conflict, so match it exactly
# (untracked result files are not touched by this)
!git fetch origin && git reset --hard origin/main
sys.path.insert(0, REPO)

!pip -q install vllm

In [ ]:
# 2. GitHub push access (once per runtime; the token is never stored or printed)
from getpass import getpass
import subprocess

token = getpass('GitHub token (repo scope): ')
subprocess.run(['git', 'config', 'user.name', 'Ngoc Doan'], cwd=REPO)
subprocess.run(['git', 'config', 'user.email', 'pnngocdoan@users.noreply.github.com'], cwd=REPO)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/pnngocdoan/agent-history-effects.git'], cwd=REPO)
del token

def push(msg):
    r = subprocess.run(f'git add results && git commit -m "{msg}" && git pull --rebase --autostash && git push',
                       shell=True, cwd=REPO, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip().splitlines()
    print(out[-1] if out else 'nothing to push')

In [ ]:
# 3. background saver: pushes results every 10 minutes, so a dead runtime loses at most 10 minutes
import threading, time

def _saver():
    while not _stop.is_set():
        _stop.wait(600)
        push('autosave: follow-up arms in progress')

_stop = threading.Event()
threading.Thread(target=_saver, daemon=True).start()
print('autosave armed (every 10 min)')

In [ ]:
# 4. THE RUNS: per model, serve it once, then run both arms. Rerunning only fills in what is missing.
import time
from pathlib import Path
import serve

MODELS = [('Qwen/Qwen3-1.7B',   'qwen3-1.7b'),
          ('Qwen/Qwen3-4B',     'qwen3-4b'),
          ('Qwen/Qwen3-8B-AWQ', 'qwen3-8b')]
K, EPISODES = 5, 10
ARMS = {'praise':     ('results/multiturn-praise',     '--praise'),
        'harm-first': ('results/multiturn-harm-first', '--harm-first')}

def n_done(out):
    f = Path(out) / 'episodes.jsonl'
    return sum(1 for l in f.read_text().splitlines() if l.strip()) if f.exists() else 0

for hf, name in MODELS:
    todo = [a for a, (root, _) in ARMS.items() if n_done(f'{root}/{name}') < EPISODES]
    if not todo:
        print(f'{name}: both arms complete, skipping'); continue
    print(f'=== {name}: {todo} ===', flush=True)
    !pkill -f api_server
    time.sleep(8)
    proc, log = serve.start_server(hf, name, log_dir=f'results/multiturn-praise/{name}')
    serve.wait_for_server(proc, log=log)
    for arm in todo:
        root, flag = ARMS[arm]
        print(f'--- {name} / {arm} ---', flush=True)
        !python multiturn/run_pilot.py --model {name} --rounds {K} --episodes {EPISODES} \
            --workers 8 {flag} --out {root}/{name}
    proc.terminate()
    push(f'follow-up arms: {name}')      # saved to GitHub before the next model starts
_stop.set()
push('follow-up arms complete')
print('done: results are on GitHub')

In [ ]:
# 5. compare: junk never-read rate by round (the rule-1 break) and the important file, per arm
#    baseline = k=5 episodes already in results/multiturn; valid = executed runs only
import json
from pathlib import Path
from multiturn.review import check, round_counts

def load(path, k=K):
    f = Path(path) / 'episodes.jsonl'
    rows = [json.loads(l) for l in f.read_text().splitlines() if l.strip()] if f.exists() else []
    return [r for r in rows if r['rounds'] == k]

ARM_DIRS = {'baseline':   'results/multiturn',
            'praise':     'results/multiturn-praise',
            'harm-first': 'results/multiturn-harm-first'}

for _, name in MODELS:
    print(f'\n===== {name} =====')
    print('junk deleted without reading, % of junk files present, by round'
          '   (in harm-first the important file is in r_0)')
    for arm, root in ARM_DIRS.items():
        rows = [r for r in load(f'{root}/{name}') if check(r)['erosion'] != 'nil']
        if not rows:
            print(f'  {arm:<11} no valid episodes'); continue
        never = [0] * (K + 1); present = [0] * (K + 1)
        for r in rows:
            for i, x in enumerate(round_counts(r)):
                never[i] += x['never']; present[i] += x['present']
        pct = '  '.join(f'r_{i}: {100 * n / p:3.0f}%' if p else f'r_{i}:   - ' for i, (n, p) in enumerate(zip(never, present)))
        hd = sum(check(r)['harm_done'] for r in rows)
        print(f'  {arm:<11} {pct}   | important file deleted {hd}/{len(rows)} valid')